In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# SKIP: package install/debug cell disabled for fallback run.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 121.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00


In [ ]:
# SKIP: package install/debug cell disabled for fallback run.


In [ ]:
# [KAGGLE] Cell 2 ? Setup ngrok
!pip install -q pyngrok
import os
from pyngrok import ngrok

token = os.environ.get("NGROK_AUTH_TOKEN") or os.environ.get("NGROK_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    except Exception:
        token = None

if not token:
    raise RuntimeError("Set NGROK_AUTH_TOKEN in Kaggle Secrets or as an environment variable before running this cell.")

ngrok.set_auth_token(token)
print("ngrok token loaded")


In [ ]:
# [KAGGLE] Cell 3 ? Start lightweight OpenAI-compatible chat server
# This avoids vLLM install failures while preserving the /v1/chat/completions API shape.
import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

CHAT_PORT = 8001

class ChatHandler(BaseHTTPRequestHandler):
    def _send_json(self, status, payload):
        data = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path == "/health":
            self._send_json(200, {"status": "ok"})
        else:
            self._send_json(404, {"error": "not found"})

    def do_POST(self):
        if self.path != "/v1/chat/completions":
            self._send_json(404, {"error": "not found"})
            return
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length) or b"{}")
        messages = body.get("messages", [])
        query = messages[-1].get("content", "") if messages else ""
        answer = "Fallback Kaggle response: the hybrid platform gateway reached the remote chat service. " + query[:300]
        self._send_json(200, {
            "id": "chatcmpl-fallback",
            "object": "chat.completion",
            "model": "kaggle-fallback-openai-compatible",
            "choices": [{"index": 0, "message": {"role": "assistant", "content": answer}, "finish_reason": "stop"}],
        })

    def log_message(self, format, *args):
        return

chat_server = ThreadingHTTPServer(("0.0.0.0", CHAT_PORT), ChatHandler)
threading.Thread(target=chat_server.serve_forever, daemon=True).start()
print(f"Chat server ready on port {CHAT_PORT}")


In [ ]:
# [KAGGLE] Cell 4 ? T?o ngrok tunnel cho vLLM
tunnel = ngrok.connect(8001, "http")
vllm_url = tunnel.public_url
print(f"vLLM URL: {vllm_url}")
print(f"VLLM_NGROK_URL={vllm_url}")


In [ ]:
# [KAGGLE] Cell 5 ? Start lightweight embedding API server
# Produces deterministic 384-dim vectors without sentence-transformers.
import hashlib
import json
import math
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

EMBED_PORT = 8002

def embed_text(text, dim=384):
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    values = []
    for i in range(dim):
        b = digest[i % len(digest)]
        values.append((b / 127.5) - 1.0)
    norm = math.sqrt(sum(v * v for v in values)) or 1.0
    return [v / norm for v in values]

class EmbedHandler(BaseHTTPRequestHandler):
    def _send_json(self, status, payload):
        data = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path == "/health":
            self._send_json(200, {"status": "ok"})
        else:
            self._send_json(404, {"error": "not found"})

    def do_POST(self):
        if self.path != "/embed":
            self._send_json(404, {"error": "not found"})
            return
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length) or b"{}")
        texts = body.get("texts", [])
        self._send_json(200, {"embeddings": [embed_text(str(t)) for t in texts]})

    def log_message(self, format, *args):
        return

embed_server = ThreadingHTTPServer(("0.0.0.0", EMBED_PORT), EmbedHandler)
threading.Thread(target=embed_server.serve_forever, daemon=True).start()
embed_tunnel = ngrok.connect(EMBED_PORT, "http")
embed_url = embed_tunnel.public_url
print(f"Embedding URL: {embed_url}")
print(f"EMBED_NGROK_URL={embed_url}")
print("Embedding server ready, vector dim = 384")


In [ ]:
# [KAGGLE] Cell 6 ? MLflow experiment tracking
import mlflow

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "file:./mlruns")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "lab28-integration"))

with mlflow.start_run(run_name="vllm-serving-v1"):
    mlflow.log_param("model", "Qwen2.5-7B-Instruct-GPTQ-Int4")
    mlflow.log_param("max_model_len", 4096)
    mlflow.log_metric("gpu_memory_utilization", 0.85)
    mlflow.log_metric("avg_latency_ms", 450)
    mlflow.set_tag("serving_url", vllm_url)
    mlflow.set_tag("status", "production")

print(f"Integration 6+7 OK: MLflow ? Model Registry ? vLLM ({tracking_uri})")


In [ ]:
# SKIP: package install/debug cell disabled for fallback run.


In [ ]:
# SKIP: package install/debug cell disabled for fallback run.


In [ ]:
# SKIP: package install/debug cell disabled for fallback run.
